# Chapter 10 Hands-On Coding Exam — Defensive Clinical Pipeline & Validation Suite (100 Points)

**Book Reference:** Chapter 10, Section 10.7 (Hands-On Coding Exam)  
**Suggested Time:** 35–45 minutes (Open Book, Closed Solution)

| Task | Defensive Programming / Validation Function | Points |
| :--- | :--- | :---: |
| **Task 10.1** | `parse_patient_record(raw_dict)` (Required keys, float cast & clinical bounds) | 30 |
| **Task 10.2** | `batch_validate_records(raw_records)` (Crash-proof batch quarantine log) | 35 |
| **Task 10.3** | `compute_safe_odds_ratio(a, b, c, d)` (Strict non-negative `int` & zero-cell guard) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q sharukh-tutor
import sharukh_tutor as tutor

tutor.activate(chapter="ch10", notebook="coding_exam")


---
## Exam Task 10.1 (30 Points): Defensive Patient Record Parser

Write `parse_patient_record(raw_dict)` that validates and normalizes a single patient intake dictionary:
1. Verify `raw_dict` is a `dict` containing all three required keys: `"patient_id"`, `"temp_c"`, and `"hb_g_dl"` (raise `ValueError` if any key is missing).
2. Verify `patient_id` is a non-empty string after `.strip()` (raise `ValueError` if `None` or blank).
3. Convert `temp_c` and `hb_g_dl` to `float` (reject booleans and raise `ValueError` on any non-numeric input).
4. Validate physiological bounds: `30.0 <= temp_c <= 45.0` and `2.0 <= hb_g_dl <= 25.0` (raise `ValueError` if out of range).
5. Return the clean typed dictionary `{"patient_id": pid, "temp_c": temp_c, "hb_g_dl": hb_g_dl}`.

In [ ]:
def parse_patient_record(raw_dict):
    """Validate patient record dict and return clean typed dict, or raise ValueError."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    if not isinstance(raw_dict, dict):
        raise ValueError("Record must be a dictionary")
    for key in ("patient_id", "temp_c", "hb_g_dl"):
        if key not in raw_dict:
            raise ValueError(f"Missing required key: {key}")
    raw_pid = raw_dict["patient_id"]
    if not isinstance(raw_pid, str) or not raw_pid.strip():
        raise ValueError("patient_id must be a non-empty string")
    pid = raw_pid.strip()
    raw_temp = raw_dict["temp_c"]
    raw_hb = raw_dict["hb_g_dl"]
    if isinstance(raw_temp, bool) or isinstance(raw_hb, bool):
        raise ValueError("Clinical vitals cannot be boolean")
    try:
        temp_c = float(raw_temp)
        hb_g_dl = float(raw_hb)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"Non-numeric vital sign: {exc}")
    if not (30.0 <= temp_c <= 45.0):
        raise ValueError(f"temp_c out of range [30.0, 45.0]: {temp_c}")
    if not (2.0 <= hb_g_dl <= 25.0):
        raise ValueError(f"hb_g_dl out of range [2.0, 25.0]: {hb_g_dl}")
    return {"patient_id": pid, "temp_c": temp_c, "hb_g_dl": hb_g_dl}
    ### END SOLUTION


---
## Exam Task 10.2 (35 Points): Resilient Batch Record Validator

Write `batch_validate_records(raw_records)` that processes a list of raw patient dictionaries **without crashing**:
1. Loop over `raw_records` and call `parse_patient_record(record)` inside `try / except ValueError as exc:`.
2. Append every valid cleaned dictionary to `valid_list`.
3. Append a descriptive error message string (e.g., `str(exc)`) to `rejected_errors` whenever `ValueError` is raised.
4. Return the 2-tuple `(valid_list, rejected_errors)`.

In [ ]:
def batch_validate_records(raw_records):
    """Validate all records via parse_patient_record; return (valid_list, rejected_errors)."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    valid_list = []
    rejected_errors = []
    for record in raw_records:
        try:
            cleaned = parse_patient_record(record)
        except ValueError as exc:
            rejected_errors.append(str(exc))
        else:
            valid_list.append(cleaned)
    return (valid_list, rejected_errors)
    ### END SOLUTION


---
## Exam Task 10.3 (35 Points): Strictly Validated $2 \times 2$ Odds Ratio

Write `compute_safe_odds_ratio(a, b, c, d)` for a $2 \times 2$ contingency table:
1. Validate that **all four** inputs (`a`, `b`, `c`, `d`) are non-negative `int` values and **not** `bool` (`isinstance(True, int)` is `True` in Python!). Raise `ValueError` if any input is negative, a `float`, a `bool`, a string, or `None`.
2. If `b == 0` or `c == 0`, return `None`.
3. Otherwise return `round((a * d) / (b * c), 2)`.

In [ ]:
def compute_safe_odds_ratio(a, b, c, d):
    """Validate non-negative ints (not bool); return OR (2 decimals) or None."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    for val in (a, b, c, d):
        if isinstance(val, bool) or not isinstance(val, int) or val < 0:
            raise ValueError(f"Invalid 2x2 cell count: {val!r}")
    if b == 0 or c == 0:
        return None
    return round((a * d) / (b * c), 2)
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch10_exam_01": parse_patient_record,
    "ch10_exam_02": batch_validate_records,
    "ch10_exam_03": compute_safe_odds_ratio,
}

tutor.check_exam("ch10", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 10.1 (Unified `ValueError` Contract):** Accessing `raw_dict["hb_g_dl"]` without checking key presence first leaks `KeyError`, and calling `float(None)` leaks `TypeError`. Converting all validation failures into `ValueError` lets Task 10.2 catch a single clean exception type.
- **Task 10.2 (Quarantine Instead of Crash):** Wrapping `parse_patient_record(record)` inside `try / except ValueError as exc:` isolates corrupt rows into `rejected_errors` while keeping valid rows flowing smoothly into `valid_list`.
- **Task 10.3 (The `bool` Subclass Trap):** In Python, `bool` inherits from `int`, so `isinstance(True, int)` evaluates to `True`! Always check `isinstance(val, bool)` *before* `isinstance(val, int)` when validating integer table counts.